# EDA и подготовка данных: Bank Marketing
**Команда:** _название_  |  **Участники:** Камшат, Алмас, Едиге

> Это **скелет** для ориентировки. Структура и подсказки готовы, а выводы, интерпретации и решения вы пишете сами и объясняете их своими словами. Места, которые нужно заполнить, помечены **TODO**. Не забудьте: каждый должен уметь объяснить любую строку.

**Задача:** бинарная классификация, предсказываем `y` (подписал ли клиент срочный депозит).
**Гипотеза:** _впишите ту, что указали в proposal_.

## 1. Импорты и загрузка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

# Файл bank-full.csv с UCI. Разделитель - точка с запятой
df = pd.read_csv("bank-full.csv", sep=";")
df.shape

## 2. Первичный обзор

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Что заметили (TODO):** сколько строк и столбцов? какие типы? есть ли что-то странное в `describe()` (например, отрицательные значения `balance`, значение `-1` в `pdays`)? Объясните, что это значит.

## 3. Целевая переменная и дисбаланс классов

In [ ]:
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True).round(3))

sns.countplot(data=df, x="y")
plt.title("Распределение целевой переменной y")
plt.show()

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

**TODO:** почему при таком дисбалансе одной accuracy недостаточно? Какую долю accuracy получит модель, которая всегда отвечает «no»?

## 4. Анализ пропусков

В этом датасете пропуски записаны словом **"unknown"**, а не NaN. Проверим оба варианта.

In [ ]:
print("Настоящие NaN:", df.isna().sum().sum())

unknown_counts = (df == "unknown").sum()
unknown_share = (unknown_counts / len(df) * 100).round(2)
unknown_df = pd.DataFrame({"unknown_count": unknown_counts, "unknown_%": unknown_share})
unknown_df[unknown_df["unknown_count"] > 0].sort_values("unknown_count", ascending=False)

**TODO (решение и обоснование):** в каких столбцах есть «unknown» и сколько? Что вы с ними делаете (оставить как отдельную категорию, заменить на самое частое значение, что-то ещё) и **почему**? Подумайте про `poutcome`: почему там «unknown» так много?

> Подсказка: любая замена, которая считается по данным (например, «самое частое значение»), может создать утечку, если считать её до split. Отдельная категория «unknown» таких проблем не создаёт.

## 5. Числовые признаки: распределения и скошенность

In [ ]:
num_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

print(df[num_cols].skew().round(2))

df[num_cols].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

**TODO:** какие признаки сильно скошены? Будете ли вы их преобразовывать (например, `np.log1p`)? Почему да или нет? Какие модели от этого выигрывают (подсказка: KNN и логистическая регрессия чувствительны к масштабу, деревья нет)?

## 6. Выбросы

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["age", "balance", "campaign"]):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

**TODO:** выбросы есть? Это ошибки данных или реальные значения (например, отрицательный `balance` может быть реальным долгом)? Решите, что делаете, и обоснуйте. Не удаляйте выбросы без причины.

## 7. Корреляции

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляция числовых признаков")
plt.show()

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

**TODO:** есть ли сильно связанные признаки? Какие, по вашему мнению, окажутся информативными?

## 8. Связь категориальных признаков с целью

In [ ]:
# Доля клиентов, подписавшихся на депозит, по категориям
df["y_bin"] = (df["y"] == "yes").astype(int)

def plot_yes_rate(col, order=None):
    rate = df.groupby(col)["y_bin"].mean().sort_values(ascending=False)
    if order is not None:
        rate = rate.reindex(order)
    rate.plot(kind="bar", figsize=(8, 4))
    plt.ylabel("Доля yes")
    plt.title(f"Доля подписавшихся по признаку {col}")
    plt.show()

plot_yes_rate("month")

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

In [ ]:
plot_yes_rate("poutcome")

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

In [ ]:
# TODO: постройте ещё один график для job (или marital / education / contact)

**Интерпретация:** _впишите 1–2 предложения: что показывает график и почему это важно._

> Итого по заданию нужно **минимум 6 графиков** с интерпретациями. Сейчас у вас: дисбаланс, гистограммы, boxplot, heatmap, month, poutcome. Добавьте 1–2 для запаса.

## 9. Feature engineering и кодирование

**TODO:** выберите решения и **объясните каждое письменно** (минимум одно). Идеи:
- `was_contacted_before = (pdays != -1)`, потому что `-1` означает «ранее не контактировали»
- группы по возрасту
- бинарные yes/no признаки (`default`, `housing`, `loan`) в 0/1
- one-hot кодирование для `job`, `marital`, `contact`, `poutcome`
- порядковое кодирование для `education`
- что делаем с `duration` (она известна только после звонка, поэтому для реального прогноза её обычно убирают; обоснуйте решение)

Ниже минимальный пример, который можно изменить и расширить.

In [ ]:
data = df.copy()

# Целевая переменная в 0/1
data["y"] = (data["y"] == "yes").astype(int)
data = data.drop(columns=["y_bin"])

# Новый признак: был ли клиент контактирован раньше
data["was_contacted_before"] = (data["pdays"] != -1).astype(int)

# TODO: решение по duration (удалить или оставить) и почему
# data = data.drop(columns=["duration"])

# TODO: бинарные признаки, возрастные группы и т.д.

**Обоснование (TODO):** почему вы создали именно эти признаки и приняли такие решения по кодированию?

In [ ]:
# Кодирование категориальных признаков (пример: one-hot)
# TODO: проверьте, какие столбцы остались категориальными, и закодируйте их
cat_cols = data.select_dtypes(include="object").columns.tolist()
print("Категориальные:", cat_cols)

data = pd.get_dummies(data, columns=cat_cols, drop_first=True)
data.shape

## 10. Разделение на train и test
Требования задания: **80/20, `random_state=42`, и это последняя строка ноутбука.**
Масштабирование и всё, что «учится» на данных, делаем **после** split и только на train (в следующем ноутбуке, внутри Pipeline).

In [ ]:
X = data.drop(columns=["y"])
y = data["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)